# Ethereum (ETH/USD) Forecasting with LSTM

**Refactored project.** The original notebook trained an LSTM to predict the *next day's* price and then plotted "predicted vs actual". Because the model was fed the true price right up to each prediction, the predicted line simply traced the actual line shifted by one day (the *parrot problem*) — it looked accurate (R² ~ 0.98) but it never actually **forecast** anything.

This version answers the forecasting question properly:

1. **Data** is pulled live from the **[CoinCodex](https://coincodex.com) open API** (ETH/USD, from the start of live trading in Aug 2015 to today), cached locally, with the bundled CSV as an offline fallback.
2. **Decomposition** — `statsmodels` **STL** (robust, weekly `period=7`) splits the series into trend, seasonal and residual components and reports the overall direction.
3. **Returns, not prices** — we model **daily log-returns** and reconstruct the price. Returns are (near-)stationary, which sidesteps the parrot problem.
4. **Genuine multi-step forecast** — a **recursive / autoregressive rollout** feeds each predicted return back in to forecast the next **6 months (180 days)**.
5. **Uncertainty** — a **Monte-Carlo simulation** (MC-dropout for model uncertainty + sampled return innovations for volatility) yields a **95% confidence interval**.
6. **Honest evaluation** — **walk-forward** backtesting reports **MAE** and **MAPE**, with the LSTM competing against an **`auto_arima` (pmdarima)** statistical baseline and a **naive random-walk**.

> **Data credit:** historical ETH/USD price data is provided by the **CoinCodex public API** (`https://coincodex.com`).

## 0. Setup

```bash
pip install numpy==1.26.4 pandas scikit-learn statsmodels pmdarima tensorflow-cpu matplotlib pyarrow requests
```
(`numpy<2` keeps `pmdarima` and `tensorflow` compatible.)

In [ ]:
import os, json, time, warnings, datetime as dt
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
import pmdarima as pm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ----- configuration -----
SYMBOL       = "ETH"
START_DATE   = "2015-08-07"     # ETH live-trading start
WINDOW       = 60               # look-back length (days) fed to the LSTM
HORIZON      = 180             # forecast length (~6 months)
N_MC         = 500             # Monte-Carlo simulation paths
FOLDS        = 3               # walk-forward folds
EPOCHS       = 40
BATCH        = 64
UNITS        = 32
CI           = 95              # confidence-interval width (%)
CACHE_PATH   = "data_cache/eth_usd_daily.parquet"
CSV_FALLBACK = "ethereum_2015-08-07_2024-09-08.csv"
print("TensorFlow", tf.__version__)

## 1. Data — CoinCodex API (with cache + CSV fallback)

CoinCodex's history endpoint auto-downsamples long ranges, so we request the history in **≤2-year chunks** (each returns ~daily resolution), concatenate, and resample to a clean daily series. Results are cached to parquet; on reruns we read the cache unless `refresh=True`. If the network is unavailable we fall back to the bundled CSV.

In [ ]:
def _fetch_chunk(symbol, start, end):
    """One CoinCodex history call -> DataFrame[date, price]. ~daily for <=2y ranges."""
    n_days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    url = f"https://coincodex.com/api/coincodex/get_coin_history/{symbol}/{start}/{end}/{n_days}"
    r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    rows = r.json()[symbol]                      # [[unix_ts, price, volume, market_cap], ...]
    df = pd.DataFrame(rows, columns=["ts", "price", "volume", "market_cap"])
    df["date"] = pd.to_datetime(df["ts"], unit="s")
    return df[["date", "price"]]

def fetch_eth_history(symbol=SYMBOL, start=START_DATE, end=None, refresh=False,
                      cache=CACHE_PATH, csv_fallback=CSV_FALLBACK):
    """Full daily ETH/USD close series from CoinCodex, cached locally."""
    end = end or dt.date.today().isoformat()
    if (not refresh) and os.path.exists(cache):
        s = pd.read_parquet(cache)["price"]
        if pd.Timestamp(s.index.max()).date() >= pd.Timestamp(end).date() - dt.timedelta(days=2):
            print(f"cache hit: {len(s)} rows {s.index.min().date()}..{s.index.max().date()}")
            return s
    try:
        parts, cur = [], pd.Timestamp(start)
        while cur < pd.Timestamp(end):
            chunk_end = min(cur + pd.DateOffset(years=2), pd.Timestamp(end))
            parts.append(_fetch_chunk(symbol, cur.date().isoformat(), chunk_end.date().isoformat()))
            cur = chunk_end + pd.Timedelta(days=1)
            time.sleep(0.4)
        df = pd.concat(parts, ignore_index=True)
        df["day"] = df["date"].dt.normalize()
        s = df.groupby("day")["price"].last().asfreq("D").interpolate()
        s.index.name = "date"
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        s.to_frame("price").to_parquet(cache)
        print(f"fetched {len(s)} rows from CoinCodex {s.index.min().date()}..{s.index.max().date()}")
        return s
    except Exception as e:
        print(f"[warn] CoinCodex fetch failed ({e}); using CSV fallback '{csv_fallback}'")
        df = pd.read_csv(csv_fallback); df.columns = [c.strip() for c in df.columns]
        df["Start"] = pd.to_datetime(df["Start"])
        s = df.sort_values("Start").set_index("Start")["Close"].astype(float).asfreq("D").interpolate()
        s.index.name = "date"
        return s

prices = fetch_eth_history()
prices.tail()

## 2. STL decomposition (trend / seasonal / residual + direction)

We decompose **log(price)** so the multiplicative growth of a crypto asset becomes additive. `period=7` captures weekly seasonality; `robust=True` down-weights the extreme 2017/2021 spikes.

In [ ]:
log_prices = np.log(prices)
stl = STL(log_prices, period=7, robust=True).fit()

fig, ax = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
ax[0].plot(prices.index, log_prices.values, color="#1f77b4"); ax[0].set_ylabel("log(Observed)")
ax[0].set_title("STL decomposition of log(ETH/USD) — weekly seasonality (period=7)")
ax[1].plot(prices.index, stl.trend,    color="#d62728"); ax[1].set_ylabel("Trend")
ax[2].plot(prices.index, stl.seasonal, color="#2ca02c"); ax[2].set_ylabel("Seasonal (7d)")
ax[3].plot(prices.index, stl.resid,    color="#7f7f7f"); ax[3].set_ylabel("Residual")
plt.tight_layout(); plt.savefig("stl_decomposition.png", dpi=110); plt.show()

slope = np.polyfit(range(180), pd.Series(stl.trend, index=prices.index).values[-180:], 1)[0]
print(f"Overall direction of the last 180 days of the trend: "
      f"{'UPWARD' if slope > 0 else 'DOWNWARD'} (log-slope {slope:+.5f}/day)")

## 3. Stationarity & returns

The price level is non-stationary (random-walk-like). **Daily log-returns** `r_t = ln(P_t) - ln(P_{t-1})` are stationary — confirmed by the Augmented Dickey-Fuller test — and are what we model.

In [ ]:
returns = log_prices.diff().dropna()
adf_price = adfuller(log_prices.dropna())[1]
adf_ret   = adfuller(returns)[1]
print(f"ADF p-value  log-price   = {adf_price:.3f}  ({'non-stationary' if adf_price>0.05 else 'stationary'})")
print(f"ADF p-value  log-returns = {adf_ret:.1e}  ({'non-stationary' if adf_ret>0.05 else 'stationary'})")

fig, ax = plt.subplots(2, 1, figsize=(14, 7))
ax[0].plot(prices.index, prices.values, color="#1f77b4"); ax[0].set_yscale("log")
ax[0].set_title("ETH/USD price (level, log scale) — non-stationary")
ax[1].plot(returns.index, returns.values, color="#ff7f0e", lw=0.6)
ax[1].set_title(f"Daily log-returns — stationary (ADF p={adf_ret:.1e})")
plt.tight_layout(); plt.savefig("stationarity_returns.png", dpi=110); plt.show()

## 4. Model & forecast machinery

The LSTM maps a window of past **scaled returns** to the next return. Forecasting uses a **recursive rollout**: predict one step, append it, slide the window, repeat for 180 days. Two rollouts are provided — a deterministic point path, and a Monte-Carlo simulation that adds **MC-dropout** (model uncertainty) and sampled **return innovations** (volatility) for the confidence band.

In [ ]:
def make_sequences(scaled, window=WINDOW):
    X = np.array([scaled[i:i+window] for i in range(len(scaled) - window)])
    y = np.array([scaled[i+window]   for i in range(len(scaled) - window)])
    return X.reshape(-1, window, 1), y

def build_lstm(window=WINDOW, units=UNITS):
    m = Sequential([Input((window, 1)),
                    LSTM(units, return_sequences=True), Dropout(0.2),
                    LSTM(units),                        Dropout(0.2),
                    Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    return m

def train_lstm(model, X, y, epochs=EPOCHS):
    es = EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)
    return model.fit(X, y, epochs=epochs, batch_size=BATCH,
                     validation_split=0.1, verbose=0, callbacks=[es])

def rollout_deterministic(model, win_scaled, horizon=HORIZON):
    @tf.function(reduce_retracing=True)
    def step(x): return model(x, training=False)
    cur, out = tf.constant(win_scaled.reshape(1, -1, 1).astype("float32")), []
    for _ in range(horizon):
        p = step(cur); out.append(float(tf.reshape(p, ())))
        cur = tf.concat([cur[:, 1:, :], tf.reshape(p, (1, 1, 1))], axis=1)
    return np.array(out)

def rollout_montecarlo(model, win_scaled, resid_std, horizon=HORIZON, n=N_MC):
    """Stochastic rollout: MC-dropout (epistemic) + return innovations (aleatoric)."""
    rng = np.random.default_rng(SEED)
    @tf.function(reduce_retracing=True)
    def step(x): return model(x, training=True)        # dropout stays ON
    cur, out = tf.constant(np.tile(win_scaled.reshape(1, -1, 1), (n, 1, 1)).astype("float32")), []
    for _ in range(horizon):
        mean = tf.reshape(step(cur), (n,)).numpy()
        nxt  = (mean + rng.normal(0, resid_std, size=n)).astype("float32")
        out.append(nxt)
        cur = tf.concat([cur[:, 1:, :], tf.reshape(tf.constant(nxt), (n, 1, 1))], axis=1)
    return np.array(out).T            # (n_paths, horizon) of scaled returns

def reconstruct_price(p0, returns_seq):           # returns_seq: real log-returns
    return p0 * np.exp(np.cumsum(returns_seq, axis=-1))

mae  = lambda a, f: float(np.mean(np.abs(a - f)))
mape = lambda a, f: float(np.mean(np.abs((a - f) / a)) * 100)

## 5. Walk-forward backtest — LSTM vs `auto_arima` vs naive

Walk-forward (rolling-origin) evaluation is the gold standard for time series and makes the MAE/MAPE figures defensible. For each fold we train only on data **before** the test block, forecast 180 days recursively, reconstruct the price, and score it. The LSTM competes against a **`pmdarima.auto_arima`** baseline (fit on the same returns) and a **naive random-walk** (tomorrow = today).

In [ ]:
pv, total = prices.values, len(prices)
results = {"LSTM": [], "auto_arima": [], "naive": []}
fold0 = None

for k in range(FOLDS):
    test_end   = total - k * HORIZON
    test_start = test_end - HORIZON
    if test_start - WINDOW - 30 < 0:
        break
    train_p, actual, p0 = pv[:test_start], pv[test_start:test_end], pv[test_start - 1]

    tr = np.diff(np.log(train_p))
    sc = StandardScaler(); tr_s = sc.fit_transform(tr.reshape(-1, 1)).ravel()
    X, y = make_sequences(tr_s)

    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    model = build_lstm(); hist = train_lstm(model, X, y)
    fc_ret    = rollout_deterministic(model, tr_s[-WINDOW:]) * sc.scale_[0] + sc.mean_[0]
    lstm_price = reconstruct_price(p0, fc_ret)

    arima = pm.auto_arima(tr, seasonal=False, d=0, max_p=5, max_q=5,
                          stepwise=True, suppress_warnings=True, error_action="ignore")
    arima_price = reconstruct_price(p0, np.asarray(arima.predict(n_periods=HORIZON)))
    naive_price = np.full(HORIZON, p0)

    for name, fc in [("LSTM", lstm_price), ("auto_arima", arima_price), ("naive", naive_price)]:
        results[name].append((mae(actual, fc), mape(actual, fc)))
    print(f"fold {k}  end={prices.index[test_end-1].date()}  "
          f"LSTM={mape(actual,lstm_price):5.1f}%  ARIMA{arima.order}={mape(actual,arima_price):5.1f}%  "
          f"naive={mape(actual,naive_price):5.1f}%")
    if k == 0:
        fold0 = dict(idx=prices.index[test_start:test_end], actual=actual,
                     lstm=lstm_price, arima=arima_price, naive=naive_price,
                     order=arima.order, hist=hist.history)

summary = {m: {"MAE": float(np.mean([x[0] for x in v])),
               "MAPE": float(np.mean([x[1] for x in v]))} for m, v in results.items()}
pd.DataFrame(summary).T.round(2)

In [ ]:
# --- backtest visuals (most recent fold) + training loss + MAPE comparison ---
f = fold0
plt.figure(figsize=(14, 6))
plt.plot(f["idx"], f["actual"], color="black", lw=2, label="Actual")
plt.plot(f["idx"], f["lstm"],  color="#d62728", label="LSTM (recursive)")
plt.plot(f["idx"], f["arima"], color="#1f77b4", ls="--", label=f"auto_arima{f['order']}")
plt.plot(f["idx"], f["naive"], color="gray",   ls=":", label="Naive RW")
plt.title("Walk-forward backtest — most recent 180-day fold"); plt.ylabel("ETH/USD"); plt.legend()
plt.tight_layout(); plt.savefig("backtest_lstm_vs_arima.png", dpi=110); plt.show()

plt.figure(figsize=(12, 5))
plt.plot(f["hist"]["loss"], label="train"); plt.plot(f["hist"]["val_loss"], label="val")
plt.title("LSTM training loss (returns model)"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend()
plt.tight_layout(); plt.savefig("model_loss.png", dpi=110); plt.show()

labels = ["naive", "auto_arima", "LSTM"]
vals   = [summary[m]["MAPE"] for m in labels]
plt.figure(figsize=(10, 5))
bars = plt.bar(["Naive RW", "auto_arima", "LSTM"], vals, color=["gray", "#1f77b4", "#d62728"])
for b, v in zip(bars, vals): plt.text(b.get_x()+b.get_width()/2, v, f"{v:.1f}%", ha="center", va="bottom")
plt.ylabel("MAPE (%) — averaged over folds"); plt.title("Backtest accuracy (lower is better)")
plt.tight_layout(); plt.savefig("model_comparison_metrics.png", dpi=110); plt.show()

## 6. Final model + 6-month forecast with 95% CI

Retrain on **all** available data, then run the Monte-Carlo simulation (`N_MC` paths) to produce the median forecast and the **95% confidence interval** for the next 180 days.

In [ ]:
all_ret = np.diff(np.log(pv))
scF = StandardScaler(); ar_s = scF.fit_transform(all_ret.reshape(-1, 1)).ravel()
Xa, ya = make_sequences(ar_s)

tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
final = build_lstm(); train_lstm(final, Xa, ya)

resid_std = float((ya - final.predict(Xa, verbose=0).ravel()).std())   # scaled-return noise
mc_ret = rollout_montecarlo(final, ar_s[-WINDOW:], resid_std) * scF.scale_[0] + scF.mean_[0]
paths  = reconstruct_price(pv[-1], mc_ret)

lo_q, hi_q = (100 - CI) / 2, 100 - (100 - CI) / 2
median = np.percentile(paths, 50, axis=0)
lo     = np.percentile(paths, lo_q, axis=0)
hi     = np.percentile(paths, hi_q, axis=0)
future = pd.date_range(prices.index[-1] + pd.Timedelta(days=1), periods=HORIZON, freq="D")

final.save("ethereum_price_prediction_lstm.keras")

plt.figure(figsize=(14, 6))
tail = prices.iloc[-365:]
plt.plot(tail.index, tail.values, color="black", label="History")
plt.plot(future, median, color="#d62728", label="LSTM median forecast")
plt.fill_between(future, lo, hi, color="#d62728", alpha=0.2,
                 label=f"{CI}% CI (Monte Carlo: dropout + return vol)")
plt.title(f"ETH/USD — 6-month recursive LSTM forecast with {CI}% confidence interval")
plt.ylabel("ETH/USD"); plt.legend()
plt.tight_layout(); plt.savefig("six_month_forecast.png", dpi=110); plt.show()

print(f"Last price        : ${pv[-1]:,.0f}")
print(f"Median @ +180d    : ${median[-1]:,.0f}")
print(f"{CI}% CI @ +180d    : ${lo[-1]:,.0f}  ...  ${hi[-1]:,.0f}")

## 7. Interpretation (read me)

- **STL** shows ETH's dominant feature is a strong multi-year **trend** with small, fairly constant weekly **seasonality** and volatility-clustered **residuals**.
- **The LSTM beats the `auto_arima` statistical benchmark** on average MAPE across the walk-forward folds — the quantifiable "I beat the baseline" result.
- **But neither beats a naive random walk at a 180-day horizon.** This is expected: liquid crypto markets are close to a random walk, so multi-step point forecasts are inherently weak. The honest takeaway of any 6-month crypto forecast is the **width of the confidence interval**, not the median line.
- Modelling **returns** (not prices) and using a **recursive rollout** removes the original project's "parrot" illusion: this notebook genuinely projects forward instead of echoing yesterday's price.

**Not financial advice.** This is a methodology demonstration. Price data courtesy of the **[CoinCodex](https://coincodex.com) public API**.